# ICL-PILOT: generation, TalkTag and UD annotation

Definitive ENNI study: 238 real children (190 TD, 48 SLI), 48 prompt pairs and 18 generation batches. Three pinned text models each generate 2,880 stories per stage at ten replicates: 17,280 narratives overall.

Use the complete prepared exemplars, fixed cross-story assignments and exact model-specific Stage 1 parents. Configuration and data sources are defined in `configs/generation/experiment.json`. Run-level seed 42, temperature 0.8, top-p 0.9, top-k 0 and 640 new tokens apply to both stages. Qwen thinking is disabled; the shared system sentence is folded into the user message before native chat formatting.

Jump to: [setup](#1-common-setup) | [generation](#2-generation) | [generation review](#review-generated-stories) | [CHAT conversion](#3-release-generation-model-segment-narratives-and-export-chat) | [shared preparation](#shared-annotation-preparation) | [TalkTag](#4-talktag-error-annotation) | [Batchalign UD](#batchalign-ud) | [exports](#5-qc-summary-and-exports). R analyses remain in their separate notebook.

Start with the smoke test to verify execution. Smoke outputs cannot enter scientific evaluation. See `docs/guides/colab_generation_package.md` for modes, dependencies and exports.

[Final text verification and retention](#annotation-retention)


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/shamira-venturini/ICL-PILOT.git"
try:
    import google.colab
    IS_COLAB = True
except ImportError:
    IS_COLAB = False
candidates = [Path.cwd(), *Path.cwd().parents, Path("/content/ICL-PILOT")]
REPO_DIR = next((p for p in candidates if (p / "pyproject.toml").is_file()), None)
if REPO_DIR is None:
    if not IS_COLAB:
        raise RuntimeError("Open this notebook from the accompanying ICL-PILOT checkout.")
    REPO_DIR = Path("/content/ICL-PILOT")
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
required = ["src/icl_pilot/pipeline.py", "src/icl_pilot/chat/annotation.py",
            "src/icl_pilot/generation/execution.py", "src/icl_pilot/generation/models.py",
            "src/icl_pilot/generation/inference.py", "configs/generation/models.json",
            "configs/annotation/talktag.json", "src/icl_pilot/chat/text_cleaning.py",
            "src/icl_pilot/chat/validation.py", "src/icl_pilot/chat/chatter_runtime.py",
            "configs/validation/chatter.json", "src/icl_pilot/chat/segmentation.py",
            "configs/segmentation/batchalign.json"]
missing = [name for name in required if not (REPO_DIR / name).is_file()]
if missing:
    raise RuntimeError("This checkout does not contain the notebook's updated helpers/configuration. "
                       "Use the accompanying project revision or upload that checkout. Missing: " + ", ".join(missing))
sys.path.insert(0, str(REPO_DIR / "src"))
print("Project:", REPO_DIR)

## 1. Common setup

Use `SMOKE_TEST=True` and choose `SMOKE_TEST_JOBS` to limit the number of
target-story jobs. The current small-batch default is 12 jobs: **12 TD-like +
12 SLI-conditioned narratives**. Set the count to 1 for the original one-story
check. Test outputs are labelled and are not study data. Set `SMOKE_TEST=False`
for the complete experiment. This switch applies to new generation, not saved-input routes.

Choose the route and input paths below. Keep the default `REPLICATES=None`
to reproduce the configured ten replicates. A positive integer changes only
the new run; the frozen study configuration remains unchanged. With all 48
pairs, each replicate produces 288 TD-like and 288 SLI-conditioned narratives.

For `from_csv` / `convert_only` / `review_only`, supply cohort CSVs in `GENERATION_CSVS`.
Do not select both an aggregate CSV and its constituent cohort CSVs.
Use `review_only` for a separately runnable QC report on saved outputs.
It loads no models and leaves every story and bundle in the data.
For `from_chat`, `prepare_only` or `morphosyntax_only`, supply CHAT directories in `CHAT_INPUT_DIRS`; each may contain
`stage1_td` / `stage2_sli` subdirectories or a single collection of CHAT files.
Each input label must be unique, such as `age4` or `real_enni`.
Use `from_chat` for a combined real-plus-all-generators run; include real ENNI
once. Specify collection provenance in `CHAT_INPUT_METADATA`. Every annotation
run starts with the same shared cleaning/validation policy. `prepare_only`
performs that step without loading TalkTag or UD.
The full/from_csv/from_chat routes run TalkTag followed by Batchalign 3 UD
annotation by default. To tag saved real or synthetic CHAT without generation,
segmentation or TalkTag, select `MODE="morphosyntax_only"`, run common setup,
then jump to the Batchalign UD setup/run cells. `RUN_MORPHOSYNTAX=False` skips UD.

The prompt-ready exemplar CSV and its extraction audit travel with the checkout in
`configs/generation/inputs/`. Both Colab (including VS Code) and local Jupyter
use the checkout as `DATA_ROOT`, so the accompanying project revision needs no
separate input upload. Colab outputs persist under
`MyDrive/icl_pilot_generation_exports`. An explicit `ICL_PILOT_DATA_ROOT`
environment variable can select another tree containing the same configured
relative input path. For a separate CSV upload, set `GENERATION_INPUT_UPLOAD`
to its runtime path; the input cell copies it only to a missing destination,
verifies the hash and preserves conflicting existing inputs.

Use a fresh `RUN_ID` for every execution. For independent blocks, run this
common setup and the selected block; skipped generation cells do not load a model.

To continue interrupted generation, set `RESUME_RUN` to the original run directory
(local or mounted Drive) and retain its original model, smoke-test size and replicate settings.
Run setup/model blocks in order. The model loads before saved random states are restored.
Only one kernel may write a run. For subsequent CHAT/annotation work, use
`from_csv` or `from_chat` in a fresh processing run.


In [ ]:
from datetime import datetime, timezone
from icl_pilot.pipeline import stage_flags

MODE = "full"  # full, generation_only, from_csv, convert_only, from_chat, prepare_only, morphosyntax_only, review_only
MODEL_KEY = "llama3_1_8b"  # llama3_1_8b, qwen3_8b, gemma2_9b; one model per fresh run
SMOKE_TEST = True  # Bounded test; set False for the full study run
SMOKE_TEST_JOBS = 12  # First 12 scheduled jobs: two replicates of six stories from the first pair
RUN_GENERATION, RUN_CHAT_CONVERSION, RUN_ANNOTATION = stage_flags(MODE)
RUN_MORPHOSYNTAX = RUN_ANNOTATION or MODE == "morphosyntax_only"  # Set False to stop after TalkTag
RESUME_RUN = None  # To continue: Path("/content/drive/MyDrive/icl_pilot_generation_exports/ORIGINAL_RUN_ID")
RESUME_GENERATION = RESUME_RUN is not None
REPLICATES = None  # None uses experiment.json; otherwise a positive integer
RUN_GENERATION_REVIEW = MODE in {"full", "generation_only", "from_csv", "convert_only", "review_only"}
REVIEW_JOURNALS = {}  # Optional {CSV_LABEL: Path("/saved/run/prompt_records.jsonl")}; otherwise look beside each CSV
DOWNLOAD_REVIEW_HTML = False  # Colab: download the offline review page after creating it
GENERATION_CSVS = {}  # e.g. {"age4": Path("/content/drive/MyDrive/run/4_year_old_rotating_story_generations.csv")}
CHAT_INPUT_DIRS = {}  # Multiple labelled collections; see shared preparation below
CHAT_INPUT_METADATA = {}  # Same labels: {"real_enni": {"source_type": "real", "corpus": "ENNI"}, ...}
ANNOTATION_STAGES = ["stage1_td", "stage2_sli"]
LEAN_ANNOTATION = True  # Verify final text, then remove successful-run intermediate CHAT copies
ORIGINAL_CORPUS_ARCHIVES = {}  # Optional {"ENNI": Path("/content/drive/MyDrive/ENNI.zip"), "Gillam": ...}
RETENTION_CONFIG_PATH = REPO_DIR / "configs/annotation/retention.json"

CONFIG_PATH = REPO_DIR / "configs/generation/experiment.json"
ROSTER_CSV = REPO_DIR / "configs/generation/master_frozen_roster_enni_2026-10-28.csv"
PREPARATION_CONFIG_PATH = REPO_DIR / "configs/annotation/preparation.json"
REVIEW_CONFIG_PATH = REPO_DIR / "configs/generation/review.json"
ANNOTATION_CONFIG_PATH = REPO_DIR / "configs/annotation/talktag.json"
MORPHOSYNTAX_CONFIG_PATH = REPO_DIR / "configs/annotation/batchalign.json"
MORPHOSYNTAX_ENV = Path("/content/icl_pilot_ud_env") if IS_COLAB else REPO_DIR / ".venv-morphosyntax"
MORPHOSYNTAX_MODELS_DIR = Path("/content/icl_pilot_ud_models") if IS_COLAB else REPO_DIR / ".cache/stanza_ud"
CHATTER_CONFIG_PATH = REPO_DIR / "configs/validation/chatter.json"
SEGMENTATION_CONFIG_PATH = REPO_DIR / "configs/segmentation/batchalign.json"
SEGMENTATION_ENV = Path("/content/icl_pilot_segmentation_env") if IS_COLAB else REPO_DIR / ".venv-segmentation"
CHATTER_EXECUTABLE = None  # Reuse/download pinned native CLI; optional explicit verified path
DATA_ROOT = Path(os.environ.get("ICL_PILOT_DATA_ROOT", str(REPO_DIR))).expanduser()
GENERATION_INPUT_UPLOAD = None  # Optional separate upload, e.g. Path("/content/story_unit_narratives_enni_2026-10-28.csv")
RUN_ID = ("enni_smoke_" if SMOKE_TEST and RUN_GENERATION else "enni_") + (MODEL_KEY + "_" if RUN_GENERATION else "") + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUN_OUTPUT_DIR = Path(RESUME_RUN).expanduser() if RESUME_GENERATION else REPO_DIR / "data/generated/direct_generation" / RUN_ID
if RESUME_GENERATION:
    if not RUN_GENERATION:
        raise ValueError("RESUME_RUN is only for interrupted generation; use saved-input routes for later steps.")
    RUN_ID = RUN_OUTPUT_DIR.name
ANNOTATION_ENV = Path("/content/icl_pilot_talktag_env") if IS_COLAB else REPO_DIR / ".venv-talktag"

INSTALL_DEPENDENCIES = True
SAVE_EVERY = 25
SAVE_TO_DRIVE = IS_COLAB
DRIVE_EXPORT_ROOT = Path("/content/drive/MyDrive/icl_pilot_generation_exports")
MAKE_ARCHIVE = True
DOWNLOAD_ARCHIVE = False
RELEASE_COLAB_RUNTIME = False

print({"mode": MODE, "generation": RUN_GENERATION, "chat": RUN_CHAT_CONVERSION,
       "annotation": RUN_ANNOTATION, "ud_annotation": RUN_MORPHOSYNTAX, "smoke_test": SMOKE_TEST and RUN_GENERATION, "new_run": str(RUN_OUTPUT_DIR)})

In [ ]:
if RUN_CHAT_CONVERSION and sys.version_info < (3, 10):
    raise RuntimeError("Batchalign segmentation requires Python >=3.10. Use the Colab runtime.")
if (RUN_ANNOTATION or globals().get("RUN_MORPHOSYNTAX", False)) and sys.version_info < (3, 10):
    raise RuntimeError("TalkTag/Batchalign annotation requires Python >=3.10. Use the Colab runtime.")
if RUN_GENERATION and sys.version_info < (3, 9):
    raise RuntimeError("Generation with Transformers 4.57.1 requires Python >=3.9. Use the Colab runtime.")
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--no-deps", "-e", str(REPO_DIR)], check=True)
    if RUN_GENERATION:
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "numpy", "pandas", "scipy", "tqdm",
                        "torch", "transformers==4.57.1", "accelerate", "bitsandbytes",
                        "huggingface_hub==0.36.0", "sentencepiece"], check=True)

In [ ]:
import json
from icl_pilot.pipeline import (begin_pipeline, checked_sources, record_stage,
                                convert_csv_inputs, annotation_inputs, annotation_collection_metadata, sync_run, package_run)

RESUME_GENERATION = globals().get("RESUME_GENERATION", False)
from icl_pilot.generation.recovery import sync_generation_checkpoint

drive_root = Path("/content/drive")
input_on_drive = (IS_COLAB and RUN_GENERATION and
                  (DATA_ROOT.resolve() == drive_root or drive_root in DATA_ROOT.resolve().parents))
resume_on_drive = IS_COLAB and RESUME_GENERATION and drive_root in RUN_OUTPUT_DIR.resolve().parents
saved_on_drive = IS_COLAB and any(drive_root in Path(value).expanduser().absolute().parents
    for value in [*GENERATION_CSVS.values(), *CHAT_INPUT_DIRS.values(), *globals().get("ORIGINAL_CORPUS_ARCHIVES", {}).values()])
if SAVE_TO_DRIVE or input_on_drive or resume_on_drive or saved_on_drive:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").is_dir():
        drive.mount("/content/drive")
    if not Path("/content/drive/MyDrive").is_dir():
        raise RuntimeError("Google Drive is required for the selected inputs/exports but is not mounted.")

generation_csvs = checked_sources(GENERATION_CSVS) if MODE in {"from_csv", "convert_only", "review_only"} else {}
chat_roots = checked_sources(CHAT_INPUT_DIRS, directories=True) if MODE in {"from_chat", "prepare_only", "morphosyntax_only"} else {}
RUN_OUTPUT_DIR = begin_pipeline(RUN_OUTPUT_DIR, mode=MODE, csvs=generation_csvs, chat_dirs=chat_roots,
                                smoke_test=SMOKE_TEST and RUN_GENERATION, resume=RESUME_GENERATION)
for stage, enabled in (("generation", RUN_GENERATION),
                       ("generation_review", globals().get("RUN_GENERATION_REVIEW", RUN_GENERATION or MODE in {"from_csv", "convert_only", "review_only"})), ("chat_conversion", RUN_CHAT_CONVERSION),
                       ("annotation_preparation", RUN_ANNOTATION or MODE == "prepare_only"),
                       ("annotation", RUN_ANNOTATION),
                       ("morphosyntax", globals().get("RUN_MORPHOSYNTAX", False)), ("segmentation", RUN_CHAT_CONVERSION),
                       ("chat_validation_before", RUN_CHAT_CONVERSION or RUN_ANNOTATION),
                       ("chat_validation_after", RUN_ANNOTATION)):
    if not RESUME_GENERATION:
        record_stage(RUN_OUTPUT_DIR, stage, "pending" if enabled else "skipped")

def persist_run():
    if SAVE_TO_DRIVE and RUN_OUTPUT_DIR.resolve() != (DRIVE_EXPORT_ROOT / RUN_ID).resolve():
        target = sync_run(RUN_OUTPUT_DIR, DRIVE_EXPORT_ROOT / RUN_ID)
        print("Verified persistent copy:", target)

def persist_generation_checkpoint(source):
    if SAVE_TO_DRIVE:
        sync_generation_checkpoint(source, DRIVE_EXPORT_ROOT / RUN_ID)

print("Resuming pipeline:" if RESUME_GENERATION else "New pipeline run:", RUN_OUTPUT_DIR)

In [ ]:
if RUN_GENERATION:
    import shutil
    from uuid import uuid4
    from icl_pilot.chat.annotation import sha256_file

    relative_input = Path(json.loads(CONFIG_PATH.read_text())["inputs"]["narratives_csv"])
    if relative_input.is_absolute() or ".." in relative_input.parts:
        raise ValueError("Generation input must stay inside DATA_ROOT.")
    input_csv = (DATA_ROOT / relative_input).resolve()
    if DATA_ROOT.resolve() not in input_csv.parents:
        raise ValueError("Generation input must stay inside DATA_ROOT.")
    upload = Path(GENERATION_INPUT_UPLOAD).expanduser() if GENERATION_INPUT_UPLOAD is not None else None
    input_action = "reused"
    if input_csv.exists():
        if not input_csv.is_file():
            raise FileExistsError(f"Expected an input file: {input_csv}")
        if upload is not None and upload.is_file() and sha256_file(upload) != sha256_file(input_csv):
            raise FileExistsError("Uploaded CSV differs from the existing input. Choose a fresh DATA_ROOT; "
                                  "the existing file has been preserved.")
    else:
        if upload is None or not upload.is_file():
            raise FileNotFoundError(f"Input missing: {input_csv}. Use the accompanying checkout with "
                                    "configs/generation/inputs/, or use VS Code's Upload to Colab "
                                    f"for {relative_input.name} and set GENERATION_INPUT_UPLOAD to its runtime path. "
                                    "Alternatively set DATA_ROOT to an existing input tree with the configured relative path.")
        input_csv.parent.mkdir(parents=True, exist_ok=True)
        temporary = input_csv.with_name(f".{input_csv.name}.{uuid4().hex}.copying")
        try:
            with upload.open("rb") as source, temporary.open("xb") as destination:
                shutil.copyfileobj(source, destination)
            if sha256_file(temporary) != sha256_file(upload):
                raise OSError("Generation input copy failed hash verification.")
            if input_csv.exists():
                raise FileExistsError(f"Input appeared during copying; preserved: {input_csv}")
            temporary.rename(input_csv)
        finally:
            if temporary.exists():
                temporary.unlink()
        input_action = "copied_and_verified"
    input_record = {"path": str(input_csv), "sha256": sha256_file(input_csv), "action": input_action}
    pipeline_path = RUN_OUTPUT_DIR / "pipeline_manifest.json"
    pipeline_record = json.loads(pipeline_path.read_text())
    pipeline_record.setdefault("generation_input", input_record)
    pipeline_record.setdefault("generation_input_checks", []).append(input_record)
    from icl_pilot.generation.recovery import atomic_json
    atomic_json(pipeline_path, pipeline_record)
    print("Generation input:", input_record)
    persist_run()
else:
    print("Exemplar input skipped; saved CSV/CHAT inputs will be used.")

## 2. Generation

This block validates the exemplar corpus, frozen ages and within-round
prompt/evaluation separation before loading the model. Inspect the job counts
before running the model-loading cell. The exemplar CSV path is relative to
`DATA_ROOT` (the checkout by default) and points into `configs/generation/inputs/`;
scaffolds and frozen prompts also come from `configs/generation/`.
The working text-only comparison shortlist is Llama 3.1 8B Instruct, Qwen3-8B
(non-thinking) and Gemma 2 9B IT. Select one with MODEL_KEY: llama3_1_8b,
qwen3_8b or gemma2_9b. All three use the shared loader and native chat adapters.
Run one checkpoint per fresh run; repeat the same smoke selection for each model.
Local software/template tests passed; full model inference still needs a Colab GPU test.
The vision condition is excluded from this study.
CHAT-to-CSV extraction removes annotator comments, pauses, xxx/yyy/www and &= codes
without guessing missing words or changing grammar. This notebook reads the resulting
prompt-ready CSV and validates/selects literal examples; it does not clean their text.
Leftover markers/comments fail preflight. The selected-text snapshot remains in
generation_design/prepared_exemplars.csv; removal counts belong to the extraction audit.
Both stages use those saved CSV texts; the Stage 1 parent remains literal.
Official model/tokenizer revisions are pinned in configs/generation/models.json.
Llama and Gemma require approved Hugging Face access. Supply an `HF_TOKEN`
through the runtime environment or an existing Hugging Face login; the loading cell
uses that authentication. The archived community conversion retains its original attribution.

Both reviewed prompts share the system message, ordered events, required content,
optional content, additional content constraints and full instruction list.
The exact system sentence is prepended to the user content for every model before native formatting.
The example guides wording, sentence construction, style and level of detail in both stages.
Stage 2 receives the exact Stage 1 narrative to rewrite with its selected exemplar.
Additional constraints are content-only. Age/status labels, ENNI/task labels, story/exemplar
IDs, titles and replicate IDs remain outside the templates and in validation/logs.
Story content and literal exemplar/parent text are preserved. No stage is asked to shorten its output.
The run records exact template hashes and messages. After pulling updates in an
existing session, restart the kernel and run from the top to load the new settings
and templates into a fresh run. Inspect the complete Stage 1 preview before inference.

Both stages read their output ceiling from `experiment.json`: currently **640 new
tokens per narrative**, shared across ages four through seven. Sampling also
uses shared configured values: **temperature 0.8, top-p 0.9 and top-k disabled (`top_k=0`)**
for both stages and every model condition.
The longest real story in those ages uses 568 tokens with Llama/Qwen and
598 with Gemma, including current exemplar preparation. The ceiling
excludes input tokens and permits earlier end-of-sequence stopping; it sets no
minimum or requested story length. The prompt journal records the ceiling and
actual generated-token count, terminal token and finish reason. Native stopping
IDs are preserved; formatted prompts are tokenized without adding special tokens again.
Qwen's template explicitly disables thinking. Other checkpoint/library decoding
defaults are recorded in model_runtime; common filtering/penalty policy remains pending.

Local CSV progress is saved every `SAVE_EVERY` complete jobs and at cohort
boundaries. After every completed generation call (Stage 1 or Stage 2), the
prompt/output journal and random-state checkpoint are saved; when
`SAVE_TO_DRIVE=True`, these recovery records are also synchronized to Drive.
The complete run folder is synchronized at cohort boundaries and other
persistence points. To continue an interrupted run, set `RESUME_RUN` to its
original directory and retain its model, smoke-test size and replicate settings.
Saved random states are restored after model loading, and verified completed
calls are reused. Only one kernel may write a run. See
[generation recovery](../docs/guides/generation_recovery.md). Saved CSVs can
enter conversion or annotation in a fresh processing run.

In [ ]:
if RUN_GENERATION:
    import pandas as pd
    import torch
    from icl_pilot.generation.config import prepare_colab_generation

    from icl_pilot.generation.randomness import initialise_run_randomness

    prepared = prepare_colab_generation(CONFIG_PATH, DATA_ROOT, RUN_OUTPUT_DIR,
                                        n_replicates=REPLICATES, smoke_test=SMOKE_TEST,
                                        smoke_test_jobs=SMOKE_TEST_JOBS if SMOKE_TEST else 1,
                                        model_key=MODEL_KEY, resume=RESUME_GENERATION)
    EXPERIMENT = prepared.config
    # Fresh run only, before model loading; resume restores states after loading.
    if not RESUME_GENERATION and globals().get("_SEEDED_RUN_ID") != RUN_ID:
        initialise_run_randomness(EXPERIMENT["master_seed"], torch=torch)
        _SEEDED_RUN_ID = RUN_ID
        from icl_pilot.generation.recovery import atomic_json, timestamp
        initial_manifest = json.loads(prepared.run_manifest_path.read_text())
        initial_manifest["randomness_initialization"] = {
            "master_seed": EXPERIMENT["master_seed"], "seed_policy": "run_level",
            "initialised_at_utc": timestamp(), "before_model_loading": True,
            "streams": ["python", "numpy", "torch_cpu", "torch_cuda_all"],
            "cuda_available": torch.cuda.is_available(),
        }
        atomic_json(prepared.run_manifest_path, initial_manifest)
    run_manifest_path = prepared.run_manifest_path
    run_df = prepared.run_df
    prompt_builder = prepared.prompt_builder
    MODEL_NAME, MODEL_REVISION = EXPERIMENT["model"]["id"], EXPERIMENT["model"]["revision"]
    DO_SAMPLE = EXPERIMENT["model"]["do_sample"]
    TEMPERATURE, TOP_P = EXPERIMENT["model"]["temperature"], EXPERIMENT["model"]["top_p"]
    TOP_K = EXPERIMENT["model"]["top_k"]
    print("Selected model:", EXPERIMENT["model"]["key"], MODEL_NAME, MODEL_REVISION)
    print(run_df.groupby("cohort_label", sort=False).agg(pairs=("pair_id", "nunique"),
        jobs=("target_story", "size"), rounds=("round_id", "nunique")).to_string())
    print(f"Total: {len(run_df)} TD-like + {len(run_df)} SLI-conditioned narratives")
    if SMOKE_TEST:
        print(f"Smoke test only: first {len(run_df)} scheduled jobs; do not use these outputs as study data.")
        print("The configured seed/order are retained; repeating the same setup may repeat the same outputs.")
    print("Configured replicates:", EXPERIMENT["n_replicates"], "| Prompt/evaluation separation: passed")
    print("Maximum new tokens: TD-like", EXPERIMENT["model"]["max_new_tokens_stage1"],
          "| SLI-conditioned", EXPERIMENT["model"]["max_new_tokens_stage2"])
    print("Sampling: temperature =", TEMPERATURE, "| top_p =", TOP_P, "| top_k =", TOP_K, "(0 disables top-k)")
    persist_run()
else:
    print("Generation setup skipped; saved inputs will be used.")

In [ ]:
if RUN_GENERATION:
    from icl_pilot.generation.inference import load_generation_runtime

    # One checkpoint on one GPU. Uses HF_TOKEN or the runtime's saved HF login.
    generation_runtime, runtime_metadata = load_generation_runtime(EXPERIMENT["model"])
    model, tokenizer = generation_runtime.model, generation_runtime.tokenizer
    print({key: runtime_metadata[key] for key in
           ("model_key", "model_id", "requested_revision", "gpu_name", "effective_compute_dtype",
            "native_stop_token_ids")})
    from icl_pilot.generation.execution import record_runtime_session
    runtime_session_id = record_runtime_session(run_manifest_path, runtime_metadata, resume=RESUME_GENERATION)
    # run_generation restores saved random streams after this model load.
    persist_run()


In [ ]:
def generate_from_messages(messages, *, max_new_tokens, row, stage, journal,
                           parent_record_id=None, stage1_text=None):
    return generation_runtime.generate(
        messages, max_new_tokens=max_new_tokens, row=row, stage=stage, journal=journal,
        parent_record_id=parent_record_id, stage1_text=stage1_text,
    )


In [ ]:
if RUN_GENERATION:
    example_row = run_df.iloc[0]
    packed_messages, preview = generation_runtime.render(prompt_builder.stage1_messages(example_row))
    print("Prompt template hashes:", prompt_builder.template_hashes)
    print("Instruction packaging:", EXPERIMENT["model"]["instruction_packaging"])
    print("Native template options:", EXPERIMENT["model"]["chat_template_kwargs"])
    print(preview)


In [ ]:
if RUN_GENERATION:
    from tqdm.auto import tqdm
    from icl_pilot.generation.execution import run_generation
    record_stage(RUN_OUTPUT_DIR, "generation", "running")
    try:
        paths = run_generation(prepared, generate_from_messages, run_id=RUN_ID,
            save_every=SAVE_EVERY, on_cohort_complete=lambda path: persist_run(), progress=tqdm,
            resume=globals().get("RESUME_GENERATION", False),
            on_checkpoint=globals().get("persist_generation_checkpoint"))
        generation_csvs = {path.stem.replace("_rotating_story_generations", ""): path for path in paths}
        record_stage(RUN_OUTPUT_DIR, "generation", "complete", csvs=[str(p) for p in paths])
    except BaseException:
        record_stage(RUN_OUTPUT_DIR, "generation", "interrupted; resume from completed-call checkpoint")
        raise
    finally:
        persist_run()

### Check for unchanged Stage 2 outputs

This read-only check compares saved Stage 1 and Stage 2 text after collapsing
whitespace. An unchanged result is flagged for review; it is preserved and
is not automatically resampled, discarded or edited to introduce errors.
A changed result alone does not establish DLD-like linguistic fidelity.
Likewise, no TalkTag error tags means no errors were tagged, not that
the narrative is error-free or that the clinical target was achieved.

In [ ]:
stage2_copy_summary = []
if RUN_GENERATION or MODE in {"from_csv", "convert_only", "review_only"}:
    import pandas as pd
    for label, path in generation_csvs.items():
        saved = pd.read_csv(path, dtype=str, keep_default_na=False)
        first = saved["stage1_td_story"].map(lambda text: " ".join(text.split()))
        second = saved["stage2_sli_story"].map(lambda text: " ".join(text.split()))
        unchanged = first.eq(second)
        count = int(unchanged.sum())
        stage2_copy_summary.append({"source_csv": str(path), "jobs": len(saved),
                                    "unchanged_after_whitespace": count})
        print(f"{label}: Stage 2 unchanged after whitespace normalization in {count}/{len(saved)} jobs.")
        if count:
            print("QC flag: inspect these transformations. Outputs and sampling are unchanged.")
            print(saved.loc[unchanged, ["pair_id", "target_story"]].head(10).to_string(index=False))
else:
    print("Stage comparison skipped; this route starts from CHAT files.")

<a id="review-generated-stories"></a>
### 2.1. Review generated stories — no exclusions

Run this block after generation or with saved `GENERATION_CSVS`. To review saved
outputs without loading models, converting CHAT or running annotation, choose
`MODE="review_only"`, run common setup, then jump here. Multiple CSV labels can
represent different generators or cohorts; select aggregate CSVs or their
constituent cohort files, avoiding overlapping jobs.

Every story remains in the generation, annotation and evaluation inputs. Flags
identify empty output, generation token-limit stops, Stage 2 unchanged after
whitespace normalization, possible template/role leakage, long same-word runs
and repeated phrase loops. Thresholds in `configs/generation/review.json` are
screening heuristics, not clinical norms. Ordinary child repetitions are retained.
Expected story content, exemplars and Stage 1 text are not treated as prompt
instructions. A flag is not evidence that a clinical target was achieved or missed.

Stopping reasons are read from `prompt_records.jsonl` beside each CSV, or from
explicit `REVIEW_JOURNALS` paths keyed by CSV label. Text/stage/identity and Stage 2
parent hashes must agree. Without a journal, stopping information is shown as
unknown; truncation is never guessed from length or punctuation. A natural end
token exactly at the budget does not receive a token-limit-stop flag.

The report contains **all stories**, a flagged-story CSV, flag counts by
source/generator/stage/cohort, source hashes and an offline `review.html` page.
The page supports search, filters, full Stage 1/Stage 2 text side by side, status
and notes. Download review notes as a separate CSV; they are not read by any
analysis. In Colab set `DOWNLOAD_REVIEW_HTML=True` or download the HTML through
the Files pane/Drive; open it in your browser. Reports travel with normal exports.
Rerunning this cell creates a new timestamped report and preserves prior reports.


In [ ]:
generation_review_manifest = None
if globals().get("RUN_GENERATION_REVIEW", RUN_GENERATION or MODE in {"from_csv", "convert_only", "review_only"}):
    from datetime import datetime, timezone
    from icl_pilot.generation.review import write_generation_review, DEFAULT_CONFIG as DEFAULT_REVIEW_CONFIG
    from icl_pilot.chat.annotation import sha256_file
    record_stage(RUN_OUTPUT_DIR, "generation_review", "running")
    try:
        review_output = RUN_OUTPUT_DIR / "generation_review" / datetime.now(timezone.utc).strftime("review_%Y%m%dT%H%M%S%fZ")
        generation_review_manifest = write_generation_review(generation_csvs, review_output,
            journals=globals().get("REVIEW_JOURNALS", {}),
            config_path=globals().get("REVIEW_CONFIG_PATH", DEFAULT_REVIEW_CONFIG))
        review_record = json.loads(generation_review_manifest.read_text())
        record_stage(RUN_OUTPUT_DIR, "generation_review", "complete", manifest=str(generation_review_manifest),
            manifest_sha256=sha256_file(generation_review_manifest), review_only=True, stories_excluded=0)
        print(json.dumps(review_record["summary"], indent=2))
        print("Offline review page:", review_output / "review.html")
        print("All stories and sampling are unchanged. Review notes are not applied to analyses.")
        if globals().get("DOWNLOAD_REVIEW_HTML", False):
            if IS_COLAB:
                from google.colab import files
                files.download(str(review_output / "review.html"))
            else:
                print("Open the HTML file above in your browser.")
    except BaseException:
        record_stage(RUN_OUTPUT_DIR, "generation_review", "failed_or_interrupted; inputs retained")
        raise
    finally:
        persist_run()
else:
    print("Generation review skipped; use saved generation CSVs to inspect stopping reasons and paired raw text.")


## 3. Release generation model, segment narratives and export CHAT

Batchalign 3 (pinned package 0.10.7) segments each complete CSV narrative before
CHAT utterances are written. Its English BERT word-boundary predictions are
applied to the original words/case, preserving grammatical forms and repetitions.
The isolated runtime defaults to CPU; no audio or speech-recognition models are used.

Existing orthographic preparation is logged. Predicted terminators replace prose
sentence punctuation; inline Unicode quotes do not create extra boundaries.
Protected CHAT notation stays anchored; conflicting scopes/terminators stop
conversion for review. Original CSVs and the exact Stage 1 parent stay unchanged.
Boundary actions, word offsets, punctuation changes and runtime/model hashes
are saved with each export. Failed model/coverage checks stop execution.

The next block requires Chatter before TalkTag; preservation QC and Chatter
are required again afterwards. Segmentation is not a grammatical correction or
a clinically validated boundary judgment. Review a small varied sample before
using new exports for linguistic analyses. Existing real CHAT boundaries remain unchanged.

Releasing the generator avoids retaining it during subsequent processing.


In [ ]:
import gc
for name in ("generation_runtime", "model", "tokenizer"):
    globals().pop(name, None)
gc.collect()
if "torch" in globals() and torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Generation model released; saved artifacts remain available.")

In [ ]:
if RUN_CHAT_CONVERSION:
    from icl_pilot.chat.segmentation import (
        BatchalignSegmenter, install_segmentation_runtime, load_segmentation_config)
    segmentation_settings = load_segmentation_config(SEGMENTATION_CONFIG_PATH)
    if INSTALL_DEPENDENCIES:
        segmentation_python = install_segmentation_runtime(SEGMENTATION_ENV, SEGMENTATION_CONFIG_PATH)
    else:
        segmentation_python = SEGMENTATION_ENV / ("Scripts/python.exe" if sys.platform == "win32" else "bin/python")
    segmentation_runtime = BatchalignSegmenter(segmentation_python, config_path=SEGMENTATION_CONFIG_PATH)
    print("Segmentation:", {key: segmentation_settings[key] for key in
          ("version", "model_id", "model_revision", "device", "policy")})
    record_stage(RUN_OUTPUT_DIR, "segmentation", "ready", config=segmentation_settings)
    persist_run()
else:
    print("Segmentation skipped: this mode does not convert generated CSVs.")


In [ ]:
if RUN_CHAT_CONVERSION:
    if "segmentation_runtime" not in globals():
        raise RuntimeError("Run the Batchalign segmentation setup block first.")
    record_stage(RUN_OUTPUT_DIR, "segmentation", "running")
    record_stage(RUN_OUTPUT_DIR, "chat_conversion", "running")
    try:
        # One model load processes all selected cohorts and both stages.
        chat_roots = convert_csv_inputs(generation_csvs, RUN_OUTPUT_DIR, roster_csv=ROSTER_CSV,
                                        segmenter=segmentation_runtime)
        record_stage(RUN_OUTPUT_DIR, "segmentation", "complete",
                     manifest=str(RUN_OUTPUT_DIR / "segmentation/segmentation_manifest.json"))
        record_stage(RUN_OUTPUT_DIR, "chat_conversion", "complete", roots={k: str(v) for k, v in chat_roots.items()})
        print("CHAT roots:", chat_roots)
    except BaseException:
        record_stage(RUN_OUTPUT_DIR, "segmentation", "failed_or_interrupted; review retained diagnostics")
        record_stage(RUN_OUTPUT_DIR, "chat_conversion", "failed; existing outputs retained")
        raise
    finally:
        persist_run()


<a id="shared-annotation-preparation"></a>
### 3.1. Shared preparation of real and synthetic CHAT

For the joint study run, choose `MODE="from_chat"`, run common setup, then start
here. Prepare real ENNI once alongside every generator's saved CHAT, using
unique labels. A generator root may contain cohort directories; labels and all
nested relative paths survive preparation, TalkTag and UD. Select each file once;
overlapping input directories are rejected. Use `prepare_only` to run just this
cleaning and validation block without loading annotation models.

```python
CHAT_INPUT_DIRS = {
    "real_enni": REPO_DIR / "data/inputs/enni_verified_2026-10-28/chat_4_to_7",
    "llama": Path("/content/drive/MyDrive/LLAMA_RUN/chat"),
    "qwen": Path("/content/drive/MyDrive/QWEN_RUN/chat"),
    "gemma": Path("/content/drive/MyDrive/GEMMA_RUN/chat"),
}
CHAT_INPUT_METADATA = {
    "real_enni": {"source_type": "real", "corpus": "ENNI"},
    "llama": {"source_type": "synthetic", "generator": "llama3_1_8b"},
    "qwen": {"source_type": "synthetic", "generator": "qwen3_8b"},
    "gemma": {"source_type": "synthetic", "generator": "gemma2_9b"},
}
```

This common deletion-only policy removes all old dependent tiers, inline error
judgments, replacement/correction hints, glosses, alternative transcriptions and
annotator comments. Produced forms are retained, never replaced with target
forms. Metadata, main-tier boundaries, unknown speech, fragments, repetitions,
retracing, pauses, uncertainty, emphasis, actions, overlaps and language codes
are preserved. Structural clause markers and `bch`/`trn` utterance bookkeeping
are retained. Scope delimiters are unwrapped only when no retained code needs
them; whitespace exposed by deletion is removed only where CHAT requires it.

Originals remain unchanged. Collections explicitly marked `source_type="real"`
are referenced by file identity, checksum and available CHAT PID without duplicate
original-file backups. Synthetic/unspecified collections retain original snapshots.
With `LEAN_ANNOTATION=True`, keep one original ZIP per real corpus, referenced
by checksum rather than copied into every run. For Colab, set
`ORIGINAL_CORPUS_ARCHIVES` to your saved ZIP paths; these archives are not in the
Git checkout. Preparation checks required ZIPs before annotation starts. Available
working sources are still checked against their recorded checksums. Temporary
copies support model processing and failure recovery; successful-run copies are
removed only by the final retention block below. Final annotated CHAT, text
fingerprints, source identities, deletion audits and Chatter reports travel with
the run. Failed or incomplete runs retain their working files.
Standard Chatter structural and alignment validation is required before TalkTag
can start. Optional strict cross-utterance quotation/completion checks are not
enabled for real or synthetic files, because those relationships are outside the
study outcomes. Original markers and boundaries are preserved. Failures in the
required checks retain all prepared files and audits. Real ENNI is never resegmented.


In [ ]:
if RUN_ANNOTATION or MODE == "prepare_only":
    from icl_pilot.chat import validation as chat_validation
    from icl_pilot.chat.annotation_preparation import (prepare_annotation_collections,
        preparation_reference, require_prepared_annotations, DEFAULT_CONFIG as DEFAULT_PREPARATION_CONFIG)
    record_stage(RUN_OUTPUT_DIR, "annotation_preparation", "running")
    try:
        chatter_runtime = chat_validation.ensure_chatter(
            executable=globals().get("CHATTER_EXECUTABLE"),
            config_path=globals().get("CHATTER_CONFIG_PATH"), install=INSTALL_DEPENDENCIES)
        raw_annotation_inputs = annotation_inputs(chat_roots, ANNOTATION_STAGES)
        metadata = globals().get("CHAT_INPUT_METADATA", {})
        if RUN_CHAT_CONVERSION:
            metadata = {label: {"source_type": "synthetic", "processing_run_id": RUN_ID,
                **(metadata.get(label, {})),
                **({"generator": globals().get("MODEL_KEY"), "generation_run_id": RUN_ID}
                   if RUN_GENERATION else {})} for label in chat_roots}
        collection_metadata = annotation_collection_metadata(chat_roots, raw_annotation_inputs, metadata)
        if globals().get("LEAN_ANNOTATION", False):
            from icl_pilot.chat.corpus_archives import register_archives, DEFAULT_CONFIG as DEFAULT_RETENTION_CONFIG
            required_corpora = {item.get("corpus") for item in collection_metadata.values()
                if item.get("source_type") == "real"}
            if None in required_corpora:
                raise ValueError("Real collections require an explicit corpus in CHAT_INPUT_METADATA.")
            register_archives(RUN_OUTPUT_DIR / "original_corpus_archives.json", required=required_corpora,
                project_root=REPO_DIR, paths=globals().get("ORIGINAL_CORPUS_ARCHIVES", {}),
                config_path=globals().get("RETENTION_CONFIG_PATH", DEFAULT_RETENTION_CONFIG))
        preparation_manifest = prepare_annotation_collections(raw_annotation_inputs,
            RUN_OUTPUT_DIR / "annotation_preparation",
            config_path=globals().get("PREPARATION_CONFIG_PATH", DEFAULT_PREPARATION_CONFIG),
            collection_metadata=collection_metadata, validator=chatter_runtime,
            test_only=json.loads((RUN_OUTPUT_DIR / "pipeline_manifest.json").read_text())["test_only"])
        annotation_preparation_reference = preparation_reference(preparation_manifest)
        preparation_record = require_prepared_annotations(annotation_preparation_reference)
        prepared_annotation_inputs = {label: Path(item["prepared"])
            for label, item in preparation_record["collections"].items()}
        record_stage(RUN_OUTPUT_DIR, "annotation_preparation", "complete", **annotation_preparation_reference)
        print("Prepared collections:", {key: str(path) for key, path in prepared_annotation_inputs.items()})
    except BaseException:
        record_stage(RUN_OUTPUT_DIR, "annotation_preparation", "failed_or_interrupted; review retained audits/validation")
        raise
    finally:
        persist_run()
else:
    print("Shared preparation skipped for this route.")


In [ ]:
# Required structural gate: validate selected CHAT before installing/loading TalkTag.
if RUN_CHAT_CONVERSION or RUN_ANNOTATION:
    from icl_pilot.chat import validation as chat_validation
    record_stage(RUN_OUTPUT_DIR, "chat_validation_before", "running")
    try:
        chatter_runtime = chat_validation.ensure_chatter(
            executable=globals().get("CHATTER_EXECUTABLE"),
            config_path=globals().get("CHATTER_CONFIG_PATH"),
            install=INSTALL_DEPENDENCIES)
        if RUN_ANNOTATION:
            from icl_pilot.chat.annotation_preparation import require_prepared_annotations
            if not globals().get("annotation_preparation_reference"):
                raise RuntimeError("Run the shared annotation preparation block first.")
            preparation_record = require_prepared_annotations(annotation_preparation_reference)
            chat_validation_before = preparation_record["validation"]
            before_report = Path(chat_validation_before["manifest"])
        else:
            before_report = chat_validation.validate_chat_directories(
                chat_roots, RUN_OUTPUT_DIR / "validation/before_annotation", validator=chatter_runtime)
            chat_validation_before = chat_validation.validation_reference(before_report)
        record_stage(RUN_OUTPUT_DIR, "chat_validation_before", "complete", **chat_validation_before)
        print("Chatter passed before annotation:", before_report)
    except BaseException:
        record_stage(RUN_OUTPUT_DIR, "chat_validation_before", "failed; annotation blocked")
        raise
    finally:
        persist_run()
else:
    print("CHAT validation skipped: generation-only mode has no CHAT files.")


## 4. TalkTag error annotation

This block works with the converted CHAT files above or `CHAT_INPUT_DIRS` in
`from_chat` mode. Both synthetic stages are selected by default; change
`ANNOTATION_STAGES` to select one when using a root with stage subdirectories.
Run the shared preparation block first, including in `from_chat` mode.
All selected collections enter one TalkTag process with separate labels.

The checked source commit/options are in `configs/annotation/talktag.json`.
They specify a **new portable run**, the configured upstream source.
TalkTag is installed in an isolated environment, leaving generation dependencies
unchanged. No GitHub token is needed for its public source repository. If model
access requires credentials, supply `HF_TOKEN` through the runtime environment.

Each annotation input is copied into the new run. Wrapped CHI main tiers are
joined only in that copy. Prepared text and headers are kept for comparison; old dependent tiers and
inline linguistic annotations were removed by the shared preparation policy.
The preparation manifest and original-source lineage are retained. Exact TalkTag output is retained under `annotation/raw_talktag/`.
A deterministic postprocessor restores source capitalization and spacing in
`annotation/annotated/`, retaining inline tags at the same token boundaries.
Changed words, punctuation, order or boundaries stop processing for that file.
Final QC remains case-sensitive and checks coverage and source/tier preservation.
Failures retain outputs/logs for review. A nonzero debug limit is labelled partial.
Chatter checks are mandatory on source files and the exact input snapshot before TalkTag,
then on final postprocessed annotated files. Raw TalkTag structural diagnostics are retained.
A failed or incomplete check stops the pipeline; no validation errors are suppressed.
The source comparison also detects valid-but-changed headers, words and punctuation.
Error-tag correctness still requires linguistic review. The following block
adds fresh English UD morphology and dependencies using Batchalign 3. CLAN
measurement and feature extraction remain later stages.
To postprocess an existing retained batch without rerunning either model, see
[the recovery instructions](../docs/guides/colab_generation_package.md#conversion-and-annotation).

In [ ]:
if RUN_ANNOTATION:
    from icl_pilot.chat.validation import require_validated_report
    if not globals().get("chat_validation_before"):
        raise RuntimeError("Run the required pre-annotation CHAT validation block first.")
    require_validated_report(chat_validation_before)
    from icl_pilot.chat.annotation import (load_annotation_config, install_annotation_runtime,
                                      annotation_runtime_metadata, annotate_chat_directory)
    annotation_settings = load_annotation_config(ANNOTATION_CONFIG_PATH)
    from icl_pilot.chat.annotation_preparation import require_prepared_annotations
    if not globals().get("annotation_preparation_reference"):
        raise RuntimeError("Run shared annotation preparation before TalkTag.")
    preparation_record = require_prepared_annotations(annotation_preparation_reference)
    selected_chat_inputs = {label: Path(item["prepared"]) for label, item in preparation_record["collections"].items()}
    print({label: str(path) for label, path in selected_chat_inputs.items()})
    if INSTALL_DEPENDENCIES:
        annotation_python = install_annotation_runtime(ANNOTATION_ENV, annotation_settings)
    else:
        annotation_python = ANNOTATION_ENV / ("Scripts/python.exe" if sys.platform == "win32" else "bin/python")
    annotation_runtime = annotation_runtime_metadata(annotation_python, annotation_settings)
    print("TalkTag runtime:", annotation_runtime)

In [ ]:
if RUN_ANNOTATION:
    annotation_manifests = []
    record_stage(RUN_OUTPUT_DIR, "annotation", "running")
    try:
        # One invocation processes all selected cohorts/stages, loading TalkTag once.
        manifest_path = annotate_chat_directory(selected_chat_inputs, RUN_OUTPUT_DIR / "annotation",
            config=annotation_settings, python=annotation_python, runtime=annotation_runtime,
            validator=globals().get("chatter_runtime"),
            preparation_reference=annotation_preparation_reference)
        annotation_manifests.append(manifest_path)
        print("Annotation and QC:", manifest_path)
        statuses = [json.loads(p.read_text())["status"] for p in annotation_manifests]
        status = "partial_debug" if "partial_debug" in statuses else "complete"
        record_stage(RUN_OUTPUT_DIR, "annotation", status, manifests=[str(p) for p in annotation_manifests])
    except BaseException:
        record_stage(RUN_OUTPUT_DIR, "annotation", "failed_or_interrupted; review retained outputs")
        raise
    finally:
        persist_run()

In [ ]:
# Final annotated files must pass both preservation QC and Chatter.
if RUN_ANNOTATION:
    from icl_pilot.chat.validation import require_validated_report
    record_stage(RUN_OUTPUT_DIR, "chat_validation_after", "running")
    try:
        final_validation_reports = []
        for path in annotation_manifests:
            annotation_record = json.loads(path.read_text())
            if annotation_record["status"] not in {"annotation_complete", "partial_debug"}:
                raise RuntimeError("Annotation/source-preservation QC has not passed.")
            reference = annotation_record["chatter"]["after"]
            checked = require_validated_report(reference)
            final_validation_reports.append(reference)
            print("Chatter passed after annotation:", reference["manifest"],
                  "| CHAT files:", checked.get("files_checked"))
        record_stage(RUN_OUTPUT_DIR, "chat_validation_after", "complete", reports=final_validation_reports)
    except BaseException:
        record_stage(RUN_OUTPUT_DIR, "chat_validation_after", "failed; final files need review")
        raise
    finally:
        persist_run()


<a id="batchalign-ud"></a>
### 4.1. Batchalign 3 English UD annotation

Run after TalkTag and its final Chatter check. For saved CHAT, select
`MODE="morphosyntax_only"`, set `CHAT_INPUT_DIRS`, run common setup, then
these two cells. Generation, segmentation and TalkTag are skipped in that mode.
For fresh real-plus-synthetic annotation use `from_chat` and shared preparation
so both receive new TalkTag codes before UD. The UD-only route preserves existing
inline codes and skips shared error-code removal; use it only when that is intended.
Use the same pinned settings for real and synthetic narratives.

The isolated CPU runtime uses Batchalign 3 (package 0.10.7), Stanza 1.15.0
and checksum-pinned English models. Only tokenize/MWT/POS/lemma/dependency
processors are loaded. Retokenization and utterance splitting are disabled;
no speech recognition, grammatical correction or traditional CLAN parser runs.
Results caching is bypassed, and each selected file must be accounted for.

Inputs are copied into immutable snapshots. Existing `%mor`, `%gra`, `%umor`
and `%ugra` are removed only from prepared worker copies so annotation is fresh.
Removal counts are recorded. Final copies replace morphology/dependencies for all speakers
with UD `%mor`/`%gra`, retaining exact source words, case, punctuation, TalkTag
codes, headers, boundaries and all non-morphological tiers. CHI coverage is
recorded separately. Previous tiers remain in
the snapshots. Native Batchalign output, including its added provenance comment,
is retained separately; final files retain the original record layout.

Chatter validates prepared inputs and final outputs with full alignment checks.
Missing files/tiers, changed content or failed validation stop processing and
retain diagnostics. Utterances containing only unintelligible/non-speech
material are recorded separately when no morphology is produced. Successful
software checks do not establish linguistic tag accuracy; manual review remains
needed. No IPSyn/DSS score or feature table is calculated in this block.

Final files: `RUN_OUTPUT_DIR/morphosyntax/annotated/`. Runtime dependencies and
model storage are separate from generation/TalkTag; cached models can be reused.
See [UD annotation guide](../docs/guides/batchalign_annotation.md).


In [ ]:
if globals().get("RUN_MORPHOSYNTAX", False):
    from icl_pilot.chat.morphosyntax import (load_morphosyntax_config,
        install_morphosyntax_runtime, prepare_morphosyntax_models, annotate_morphosyntax)
    from icl_pilot.chat.validation import require_validated_report
    if RUN_ANNOTATION:
        from icl_pilot.chat.annotation_preparation import preparation_reference, checked_talktag_collections
        if len(annotation_manifests) != 1:
            raise RuntimeError("Use the single combined TalkTag manifest for this run.")
        morphosyntax_upstream = preparation_reference(annotation_manifests[0])
        morphosyntax_inputs, _, _ = checked_talktag_collections(morphosyntax_upstream)
    else:
        if MODE != "morphosyntax_only":
            raise RuntimeError("Use morphosyntax_only with saved CHAT_INPUT_DIRS.")
        morphosyntax_inputs = annotation_inputs(chat_roots, ANNOTATION_STAGES)
        morphosyntax_upstream = None  # Existing fresh TalkTag codes must remain intact on this UD-only route.
    morphosyntax_settings = load_morphosyntax_config(MORPHOSYNTAX_CONFIG_PATH)
    if INSTALL_DEPENDENCIES:
        morphosyntax_python = install_morphosyntax_runtime(MORPHOSYNTAX_ENV, MORPHOSYNTAX_CONFIG_PATH)
    else:
        morphosyntax_python = MORPHOSYNTAX_ENV / ("Scripts/python.exe" if sys.platform == "win32" else "bin/python")
    morphosyntax_model_dir = prepare_morphosyntax_models(MORPHOSYNTAX_MODELS_DIR,
        MORPHOSYNTAX_CONFIG_PATH, download=INSTALL_DEPENDENCIES)
    print("UD inputs:", {key: str(path) for key, path in morphosyntax_inputs.items()})
    print("UD versions:", {key: morphosyntax_settings[key] for key in ("version", "stanza_version", "torch_version")})
else:
    print("Batchalign UD annotation skipped.")


In [ ]:
if globals().get("RUN_MORPHOSYNTAX", False):
    from icl_pilot.chat import validation as ud_validation
    record_stage(RUN_OUTPUT_DIR, "morphosyntax", "running")
    try:
        ud_validator = ud_validation.ensure_chatter(executable=globals().get("CHATTER_EXECUTABLE"),
            config_path=CHATTER_CONFIG_PATH, install=INSTALL_DEPENDENCIES)
        morphosyntax_manifest = annotate_morphosyntax(morphosyntax_inputs, RUN_OUTPUT_DIR / "morphosyntax",
            python=morphosyntax_python, model_dir=morphosyntax_model_dir,
            config_path=MORPHOSYNTAX_CONFIG_PATH, validator=ud_validator,
            test_only=json.loads((RUN_OUTPUT_DIR / "pipeline_manifest.json").read_text())["test_only"],
            upstream_annotation=morphosyntax_upstream)
        morphosyntax_record = json.loads(morphosyntax_manifest.read_text())
        morphosyntax_output_dir = morphosyntax_manifest.parent / morphosyntax_record["annotated_output_directory"]
        record_stage(RUN_OUTPUT_DIR, "morphosyntax", "complete", manifest=str(morphosyntax_manifest),
            output=str(morphosyntax_output_dir))
        print("Validated UD-annotated CHAT:", morphosyntax_output_dir)
    except BaseException:
        record_stage(RUN_OUTPUT_DIR, "morphosyntax", "failed_or_interrupted; review retained outputs")
        raise
    finally:
        persist_run()


## 5. QC summary and exports

The run folder includes generation CSVs/journal/design when generated here,
review-only HTML/CSV/JSON reports when requested,
Batchalign boundary/runtime records, converted CHAT manifests, shared preparation
retained synthetic-original snapshots, real-source identities/hashes/PIDs,
deletion audits/collection provenance, derived annotation input snapshots, raw/postprocessed CHAT,
TalkTag logs/reports, Batchalign UD source/prepared snapshots, raw/final CHAT,
model/runtime hashes and validation/QC manifests. All travel together in the export.
Saved external input paths/hashes remain recorded; retain the original generation
run's provenance when starting from its CSV or CHAT files.

This is a processing record, not an evaluation result. Continue evaluation in
the separate Python notebook or R Markdown after annotation review and the
appropriate measurement/feature-extraction steps. Annotation ends with the
validated Batchalign UD output; traditional MOR/MEGRASP and IPSyn are not run. GPU integration testing is still
required before claiming the complete workflow has run successfully on Colab.

<a id="annotation-retention"></a>
### 3.6. Verify final text and retain one annotated copy

Run after TalkTag and, when enabled, UD. The export block also invokes this check.
Words, speaker order, main-tier boundaries and headers are checked against the
preparation baseline; UD additionally preserves all existing inline annotations.
Intended annotation removals/additions and new morphology are allowed. Every final
file is also recorded with its complete SHA-256 and per-utterance fingerprints.

Only a complete checked chain can be compacted. The block removes run-owned
source snapshots, prepared copies, raw model CHAT and (after UD) intermediate
TalkTag CHAT. It retains final CHAT, manifests, runtime records, deletion audits,
validation reports and diagnostics. External inputs, original corpus ZIPs and
generation CSVs/journals remain available. Drive synchronization removes only the
same checksum-matching intermediate files. Failed or incomplete runs are preserved.

After cleanup, use `verify_retained_annotations(reference, run_root=...)` with the
reference in `pipeline_manifest.json`. Earlier stage reports remain historical;
their deleted inputs are not needed by this final check. To rerun annotation,
start a fresh processing run from the saved input folders. To recover a particular
real original, use the ZIP/member reference with `restore_original_member`; the
documented ENNI sex-header corrections belong to the study input view.


In [ ]:
def finalize_annotation_retention():
    if not globals().get("LEAN_ANNOTATION", False) or not RUN_ANNOTATION:
        return
    from icl_pilot.chat.retention import finalize_annotation_run
    from icl_pilot.chat.corpus_archives import DEFAULT_CONFIG as DEFAULT_RETENTION_CONFIG
    stages = json.loads((RUN_OUTPUT_DIR / "pipeline_manifest.json").read_text())["stages"]
    required = ["annotation_preparation", "annotation", "chat_validation_after"]
    if globals().get("RUN_MORPHOSYNTAX", False):
        required.append("morphosyntax")
    if any(stages.get(name, {}).get("status") != "complete" for name in required):
        print("Annotation incomplete: working copies and diagnostics retained.")
        return
    if len(stages["annotation"]["manifests"]) != 1:
        raise ValueError("Retention requires the one shared TalkTag invocation.")
    reference = finalize_annotation_run(RUN_OUTPUT_DIR, annotation_manifest=stages["annotation"]["manifests"][0],
        morphosyntax_manifest=stages.get("morphosyntax", {}).get("manifest")
            if globals().get("RUN_MORPHOSYNTAX", False) else None,
        archive_paths=globals().get("ORIGINAL_CORPUS_ARCHIVES", {}), project_root=REPO_DIR,
        config_path=globals().get("RETENTION_CONFIG_PATH", DEFAULT_RETENTION_CONFIG))
    record_stage(RUN_OUTPUT_DIR, "annotation_retention", "complete", **reference)
    persist_run()
    print("Final annotations and preservation receipt:", reference["manifest"])

finalize_annotation_retention()


In [ ]:
if "finalize_annotation_retention" in globals():
    finalize_annotation_retention()
elif globals().get("LEAN_ANNOTATION", False) and RUN_ANNOTATION:
    raise RuntimeError("Run the final text-verification/retention block before exporting annotations.")
summary = json.loads((RUN_OUTPUT_DIR / "pipeline_manifest.json").read_text())
print(json.dumps(summary["stages"], indent=2))
persist_run()
if MAKE_ARCHIVE:
    archive_path = package_run(RUN_OUTPUT_DIR)
    print("Complete processing archive:", archive_path)
    persist_run()
    if DOWNLOAD_ARCHIVE:
        from google.colab import files
        files.download(str(archive_path))

if RELEASE_COLAB_RUNTIME and IS_COLAB:
    from google.colab import runtime
    runtime.unassign()